In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Step 1: Install and import required libraries
!pip install -q mediapipe opencv-python-headless

import mediapipe as mp
import cv2
import numpy as np
import os
from tqdm import tqdm




In [ ]:
DATASET_PATH = "/kaggle/input/fer2013"
if not os.path.exists(DATASET_PATH):

    DATASET_PATH = "path/to/your/fer2013/dataset"
train_dir = os.path.join(DATASET_PATH, "train")
val_dir = os.path.join(DATASET_PATH, "validation")
test_dir = os.path.join(DATASET_PATH, "test")

classes = sorted(os.listdir(train_dir))
class_to_idx = {cls_name: idx for idx, cls_name in enumerate(classes)}
print("Classes and label indices:", class_to_idx)


In [ ]:
# Step 3: Extract face landmarks from training images and save features
X_train = []
y_train = []
with mp.solutions.face_mesh.FaceMesh(static_image_mode=True, max_num_faces=1, min_detection_confidence=0.5) as face_mesh:
    for cls_name, label in class_to_idx.items():
        class_folder = os.path.join(train_dir, cls_name)
        for img_name in tqdm(os.listdir(class_folder), desc=f"Processing {cls_name}", leave=False):
            img_path = os.path.join(class_folder, img_name)
            image = cv2.imread(img_path, cv2.IMREAD_COLOR)
            if image is None:
                continue
            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            results = face_mesh.process(image_rgb)
            if results.multi_face_landmarks:
                landmarks = results.multi_face_landmarks[0].landmark
                coords = np.array([[lm.x, lm.y, lm.z] for lm in landmarks], dtype=np.float32).flatten()
                X_train.append(coords)
                y_train.append(label)
X_train = np.array(X_train, dtype=np.float32)
y_train = np.array(y_train, dtype=np.int64)
np.save("X_train.npy", X_train)
np.save("y_train.npy", y_train)
print("Train features shape:", X_train.shape, "Train labels shape:", y_train.shape)

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

X_train_raw = np.load("X_train.npy")
y_train_raw = np.load("y_train.npy")

X_train, X_val, y_train, y_val = train_test_split(
    X_train_raw, y_train_raw,
    test_size=0.2,
    random_state=42,
    stratify=y_train_raw
)

np.save("X_train.npy", X_train)
np.save("y_train.npy", y_train)
np.save("X_val.npy", X_val)
np.save("y_val.npy", y_val)



In [ ]:
X_test = []
y_test = []
with mp.solutions.face_mesh.FaceMesh(static_image_mode=True, max_num_faces=1, min_detection_confidence=0.5) as face_mesh:
    for cls_name, label in class_to_idx.items():
        class_folder = os.path.join(test_dir, cls_name)
        if not os.path.exists(class_folder):
            continue  # skip if class not present in test set
        for img_name in tqdm(os.listdir(class_folder), desc=f"Processing {cls_name}", leave=False):
            img_path = os.path.join(class_folder, img_name)
            image = cv2.imread(img_path, cv2.IMREAD_COLOR)
            if image is None:
                continue
            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            results = face_mesh.process(image_rgb)
            if results.multi_face_landmarks:
                landmarks = results.multi_face_landmarks[0].landmark
                coords = np.array([[lm.x, lm.y, lm.z] for lm in landmarks], dtype=np.float32).flatten()
                X_test.append(coords)
                y_test.append(label)
X_test = np.array(X_test, dtype=np.float32)
y_test = np.array(y_test, dtype=np.int64)
np.save("X_test.npy", X_test)
np.save("y_test.npy", y_test)
print("Test features shape:", X_test.shape, "Test labels shape:", y_test.shape)

In [ ]:
# Step 6: Load the saved numpy feature arrays (if starting from here, skip if already in memory)
X_train = np.load("X_train.npy")
y_train = np.load("y_train.npy")
X_val = np.load("X_val.npy")
y_val = np.load("y_val.npy")
X_test = np.load("X_test.npy")
y_test = np.load("y_test.npy")
print("Loaded feature shapes:", X_train.shape, X_val.shape, X_test.shape)

In [ ]:
counts = np.bincount(y_train)
print(counts)  
max_count = np.max(counts)  
sampling_dict = {}
for cls_idx, c in enumerate(counts):
    desired = max(int(0.5 * max_count), c)  
    sampling_dict[cls_idx] = desired

print(sampling_dict)


In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(
    sampling_strategy=sampling_dict,
    random_state=42
)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)
print("distribution after smote:", np.bincount(y_train_res))



In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

input_dim = X_train_res.shape[1]  
model = keras.Sequential([
    layers.Dense(256, activation='relu', input_shape=(input_dim,)),
    layers.Dropout(0.2),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(7, activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

early_stop = EarlyStopping(monitor='val_loss', patience=100, restore_best_weights=True)
checkpoint = ModelCheckpoint('best_model.keras', monitor='val_loss', save_best_only=True)
history = model.fit(X_train_res, y_train_res, validation_data=(X_val, y_val),
                    epochs=1000, batch_size=64, callbacks=[early_stop, checkpoint], verbose=1)

In [ ]:
model.save("fer_mlp_model.h5")
print("Saved model to mlp10_model.h5")


In [ ]:
from tensorflow import keras

model = keras.models.load_model("mlp10_model.h5")
model.summary()  

In [ ]:
import numpy as np

X_test = np.load("X_test.npy")
y_test = np.load("y_test.npy")

print("Test set feature dimensions:", X_test.shape)
print("Test set label dimensions:", y_test.shape)

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"accuracy of test set: {test_acc*100:.2f}%")

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)  

print("\nclassification_report:")
print(classification_report(y_test, y_pred_classes, target_names=classes))

